## Library

In [2]:
import os
import numpy as np
import torch
import librosa
from tqdm import tqdm
from transformers import Wav2Vec2FeatureExtractor, AutoModel

## so importin some stuff

## Paths and Config

In [3]:

AUDIO_DIR = "dataset/recordings/recordings"     ###Pass the audio file path inside the quotes
SAVE_DIR = "embeddings"     ###Pass the Embeddings storing path inside the quotes
MODEL_NAME = "microsoft/wavlm-base"   ##keep it as it is
SAMPLE_RATE = 16000  ### if you want you can check your audio file sample rates

os.makedirs(SAVE_DIR, exist_ok=True)

## Load Model


In [4]:
processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
### we are loading the model from huggin face so dont change anything here.

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

WavLMModel(
  (feature_extractor): WavLMFeatureEncoder(
    (conv_layers): ModuleList(
      (0): WavLMGroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x WavLMNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x WavLMNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): WavLMFeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): WavLMEncoder(
    (pos_conv_embed): WavLMPositionalConvEmbedding(
      (conv): Parametrized

## Process and store the file


In [ ]:
import re

# ... (Previous imports and model loading stay the same)

# Define the list of languages from your image
allowed_languages = [
    "dutch", "english", "arabic", "spanish", "french", 
    "mandarin", "russian", "korean", "portuguese"
]

X = []
y = [] 
file_names = []

for file in tqdm(os.listdir(AUDIO_DIR)):
    if not file.endswith((".wav", ".mp3")):
        continue

    # Extract the label (prefix) from the filename
    match = re.match(r"([a-zA-Z]+)", file)
    if match:
        label = match.group(1).lower() # Convert to lowercase for consistent matching
    else:
        label = "unknown"

    # --- NEW FILTERING LOGIC ---
    # Only proceed if the label is in our allowed list
    if label not in allowed_languages:
        continue
    # ---------------------------

    file_path = os.path.join(AUDIO_DIR, file)
    
    try:
        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)
        inputs = processor(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt")

        with torch.no_grad():
            outputs = model(**inputs)

        # Mean pooling & Normalization
        embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
        embedding = embedding / np.linalg.norm(embedding)

        # Store both embedding and label
        X.append(embedding)
        y.append(label)
        file_names.append(file)

    except Exception as e:
        print(f"Error processing {file}: {e}")

  0%|                                                                                          | 0/2138 [00:00<?, ?it/s]/home/tanish/ai_project/audio_ai_project/lib/python3.10/site-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
  6%|█████                                                                           | 134/2138 [01:56<27:37,  1.21it/s]

## Save in .npy file

In [ ]:
# Convert to arrays
X = np.array(X)
y = np.array(y)

# Save both files
np.save(os.path.join(SAVE_DIR, "embeddings.npy"), X)
np.save(os.path.join(SAVE_DIR, "labels.npy"), y)

print(f"Saved embeddings shape: {X.shape}")
print(f"Saved labels shape: {y.shape}")

In [6]:
X

array([[-0.00231228, -0.02836744, -0.01719793, ...,  0.09003277,
         0.00879641,  0.02172253],
       [-0.00853863, -0.01710543,  0.00156903, ...,  0.0703103 ,
        -0.00498133,  0.01015194],
       [-0.01960716, -0.0039057 , -0.00131781, ...,  0.1416728 ,
        -0.01551394,  0.00018731],
       ...,
       [-0.01070988, -0.04019415, -0.02066063, ...,  0.10580681,
         0.01157063,  0.00587029],
       [-0.00427142, -0.00836162, -0.01417161, ...,  0.08841698,
         0.00644572, -0.00995223],
       [-0.01648235, -0.02356419, -0.01186641, ...,  0.10917545,
         0.00347279,  0.01276528]], shape=(1166, 768), dtype=float32)

In [17]:
import pandas as pd

In [18]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,-0.002312,-0.028367,-0.017198,0.016235,0.018496,0.007039,-0.026262,-0.009193,-0.005940,-0.001336,...,0.017212,0.014880,-0.005371,0.051947,0.028333,0.004410,-0.007594,0.090033,0.008796,0.021723
1,-0.008539,-0.017105,0.001569,-0.008962,0.025258,0.016991,-0.010591,-0.026701,-0.007373,-0.001167,...,0.023181,0.014174,0.012566,0.021958,0.017272,0.001201,0.005674,0.070310,-0.004981,0.010152
2,-0.014471,-0.023379,-0.008775,-0.015173,0.028213,0.000159,-0.003754,-0.021004,-0.021017,0.006772,...,0.024621,0.014520,0.006955,0.003166,0.009606,-0.003854,0.007446,0.110976,-0.006905,0.002563
3,-0.010809,-0.027121,-0.031580,-0.012126,0.023000,0.005748,-0.022816,-0.015831,-0.005829,0.003347,...,0.025676,0.004448,-0.005587,0.021290,0.013466,0.000086,0.011169,0.106041,-0.003740,0.011549
4,-0.019607,-0.003906,-0.001318,-0.011564,0.021536,0.010595,-0.026568,-0.020814,-0.002976,0.001643,...,0.027558,0.028402,0.022577,0.018868,0.007440,-0.006221,-0.000055,0.141673,-0.015514,0.000187
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2133,-0.022561,-0.013331,-0.023535,-0.002885,0.013422,0.028471,-0.017039,-0.018845,-0.021663,0.008846,...,0.031809,0.021751,-0.000809,0.023651,0.016342,-0.018379,0.011182,0.044021,0.003873,0.013755
2134,-0.010710,-0.040194,-0.020661,-0.010114,0.026932,-0.010990,-0.018793,-0.023875,-0.027381,0.018075,...,0.036275,0.003059,-0.004014,0.027530,0.017452,-0.004799,0.016382,0.105807,0.011571,0.005870
2135,-0.004271,-0.008362,-0.014172,-0.011600,0.019928,-0.005967,-0.015194,-0.015173,0.000816,-0.004762,...,0.025735,0.013026,0.011372,0.008535,-0.006227,-0.016861,0.005849,0.088417,0.006446,-0.009952
2136,-0.008655,-0.024791,-0.038761,-0.022234,0.008418,0.034967,-0.014024,-0.018791,-0.008924,0.005572,...,0.023702,0.009628,0.004969,0.012052,0.015395,-0.008761,0.005559,0.069507,-0.006136,0.009001
